# Silver Stores

Run top to bottom in the Retail Lakehouse (WSL) kernel. Source: Bronze stores. Output: Silver stores.

Coordinate bounds and required fields come from the workbook. Blank manager names become null.

Rejections are saved in ignored error/data/stores with their original JSON record. Run the numbered notebooks in order.


In [1]:
import os  # Configure Java before starting Spark.
import shutil  # Locate the Linux Java executable.
from pathlib import Path  # Build table paths.
from uuid import uuid4  # Tag rejected rows from this run.
from pyspark.sql import SparkSession, functions as F  # Use Spark DataFrames and expressions.
from pyspark.sql.window import Window  # Count duplicate business keys.
from delta import configure_spark_with_delta_pip  # Load Delta Lake support.
java = shutil.which("java")  # Find Java inside WSL.
if java is None:  # Stop if Java is absent.
    raise RuntimeError("Java is unavailable in WSL")  # Explain the startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Avoid the Windows Java path.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Point to this project.
if not project_root.is_dir():  # Confirm the WSL mount exists.
    raise FileNotFoundError(project_root)  # Do not run against the wrong folder.
builder = (SparkSession.builder.master("local[2]")  # Keep local resource use modest.
    .appName("RetailSilverStores")  # Label this table's Spark job.
    .config("spark.sql.shuffle.partitions", "4")  # Use fewer shuffle tasks for small local tables.
    .config("spark.ui.enabled", "false")  # Avoid a second Spark UI port.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta tables.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start or reuse Spark.
print("Spark version:", spark.version)  # Confirm startup.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 19:51:57 WARN Utils: Your hostname, USMAN-ALI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/09 19:51:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/usmanali611b/.venvs/retail-lakehouse/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/usmanali611b/.ivy2.5.2/cache
The jars for the packages stored in: /home/usmanali611b/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a253a755-e337-49e5-b629-12c6a174a93c;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.1

Spark version: 4.1.1


In [3]:
table_name = 'stores'  # Name the Silver output table.
bronze_name = 'stores'  # Name its Bronze source table.
business_key = ['store_id']  # Columns that uniquely identify one row.
fields = [  # Define cleaning and type conversion for each source column.
    ('store_id', 'text'),  # Convert store_id to text.
    ('store_name', 'text'),  # Convert store_name to text.
    ('city', 'text'),  # Convert city to text.
    ('region', 'text'),  # Convert region to text.
    ('latitude', 'decimal(9,6)'),  # Convert latitude to decimal(9,6).
    ('longitude', 'decimal(9,6)'),  # Convert longitude to decimal(9,6).
    ('opened_date', 'date'),  # Convert opened_date to date.
    ('manager_name', 'optional_text'),  # Convert manager_name to optional_text.
]
references = [  # Define Silver tables needed for foreign-key checks.
]
rejection_rules = [  # Rows matching these rules go to error evidence.
    ('missing_store_id', "store_id IS NULL OR store_id = ''"),  # Check missing_store_id.
    ('duplicate_store_id', 'id_occurrences > 1'),  # Check duplicate_store_id.
    ('missing_store_name', "store_name IS NULL OR store_name = ''"),  # Check missing_store_name.
    ('missing_city', "city IS NULL OR city = ''"),  # Check missing_city.
    ('missing_region', "region IS NULL OR region = ''"),  # Check missing_region.
    ('invalid_latitude', 'latitude IS NULL OR latitude NOT BETWEEN -90 AND 90'),  # Check invalid_latitude.
    ('invalid_longitude', 'longitude IS NULL OR longitude NOT BETWEEN -180 AND 180'),  # Check invalid_longitude.
    ('invalid_opened_date', 'opened_date IS NULL'),  # Check invalid_opened_date.
]
critical_rules = [  # These rejection rules also stop the pipeline.
]
warning_rules = [  # These issues are reported but do not reject a row.
]


In [4]:
bronze_path = project_root / "lakehouse" / "bronze" / "data" / bronze_name  # Locate Bronze Delta.
bronze_df = spark.read.format("delta").load(str(bronze_path))  # Read its current committed version.
expected_columns = [name for name, kind in fields]  # Build the source schema contract.
if bronze_df.columns != expected_columns:  # Detect missing, extra, or reordered fields.
    raise ValueError(f"Unexpected Bronze columns: {bronze_df.columns}")  # Stop on schema drift.
bronze_count = bronze_df.count()  # Count every source record.
print("Bronze rows:", bronze_count)  # Show the input count.
bronze_df.printSchema()  # Show original Spark types.
bronze_df.show(3, truncate=False)  # Preview three records.


26/10/09 19:53:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Bronze rows: 6
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- opened_date: string (nullable = true)
 |-- manager_name: string (nullable = true)



[Stage 13:>                                                         (0 + 1) / 1]

+--------+-------------------+---------+------+--------+---------+-----------+------------+
|store_id|store_name         |city     |region|latitude|longitude|opened_date|manager_name|
+--------+-------------------+---------+------+--------+---------+-----------+------------+
|S001    |Lahore Gulberg     |Lahore   |Punjab|31.5204 |74.3587  |2021-03-15 |Aisha Khan  |
|S002    |Islamabad Blue Area|Islamabad|ICT   |33.6844 |73.0479  |2021-08-02 |Omar Malik  |
|S003    |Karachi Clifton    |Karachi  |Sindh |24.8607 |67.0011  |2020-11-20 |Sara Ahmed  |
+--------+-------------------+---------+------+--------+---------+-----------+------------+
only showing top 3 rows


In [5]:
def cleaned_column(name, kind):  # Convert one field according to the dictionary.
    value = F.col(name)  # Select the original Bronze field.
    if kind == "text":  # Required text stays text.
        return F.trim(value)  # Remove surrounding whitespace.
    if kind == "optional_text":  # Optional text can be absent.
        text = F.trim(value)  # Trim the value first.
        return F.when(text == "", F.lit(None)).otherwise(text)  # Make blank text null.
    target_type = "int" if kind == "optional_int" else kind  # Use Spark's integer type.
    return value.try_cast(target_type)  # Convert safely; invalid input becomes null.
clean_df = bronze_df.select(  # Keep one output row per Bronze row.
    *[cleaned_column(name, kind).alias(name) for name, kind in fields],  # Clean business fields.
    F.to_json(F.struct(*[F.col(name) for name in expected_columns])).alias("source_record"),  # Preserve original values.
)
if clean_df.count() != bronze_count:  # Conversion must never lose a row.
    raise ValueError("Cleaning changed row count")  # Stop if reconciliation fails.
clean_df.printSchema()  # Show converted Silver candidate types.


[Stage 15:===================================>                    (32 + 3) / 50]

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- latitude: decimal(9,6) (nullable = true)
 |-- longitude: decimal(9,6) (nullable = true)
 |-- opened_date: date (nullable = true)
 |-- manager_name: string (nullable = true)
 |-- source_record: string (nullable = true)



In [6]:
checked_base_df = clean_df  # Start with every cleaned source row.
for table, source_key, target_key, marker, allow_distinct in references:  # Check each foreign key.
    path = project_root / "lakehouse" / "silver" / "data" / table  # Locate the reference table.
    lookup_df = spark.read.format("delta").load(str(path)).select(target_key)  # Read just its key.
    if allow_distinct:  # A product category occurs on many products.
        lookup_df = lookup_df.distinct()  # Keep one lookup row per category.
    elif lookup_df.count() != lookup_df.distinct().count():  # Primary keys must be unique.
        raise ValueError(f"Duplicate reference keys in {table}")  # Avoid multiplying rows.
    lookup_df = lookup_df.withColumn(marker, F.lit(True))  # Mark successful matches.
    checked_base_df = checked_base_df.join(  # Preserve all source rows.
        lookup_df, checked_base_df[source_key] == lookup_df[target_key], "left"  # Match to Silver.
    ).drop(lookup_df[target_key]).fillna(False, subset=[marker])  # Remove the extra key.
    if checked_base_df.count() != bronze_count:  # A lookup must preserve source row count.
        raise ValueError(f"Reference join changed row count: {table}")  # Stop on a bad join.
print("Rows after lookups:", checked_base_df.count())  # Confirm all input rows remain.


Rows after lookups: 6


In [7]:
checked_df = checked_base_df.withColumn(  # Add duplicate-key counts.
    "id_occurrences", F.count(F.lit(1)).over(Window.partitionBy(*business_key))  # Count the cleaned key.
)
def issue_column(rules):  # Turn named validation rules into one issue string.
    return F.concat_ws("; ", *[  # Skip rules that do not match a row.
        F.when(F.coalesce(F.expr(predicate), F.lit(False)), F.lit(label))  # Evaluate each SQL predicate.
        for label, predicate in rules  # Use the rule names defined above.
    ]) if rules else F.lit("")  # Keep an empty string when no rules are defined.
checked_df = checked_df.withColumn("quality_issue", issue_column(rejection_rules))  # Reasons to reject.
checked_df = checked_df.withColumn("critical_issue", issue_column(critical_rules))  # Reasons to fail the build.
checked_df = checked_df.withColumn("warning_issue", issue_column(warning_rules))  # Nonblocking issues.
accepted_df = checked_df.filter(F.col("quality_issue") == "")  # Select clean rows.
rejected_df = checked_df.filter(F.col("quality_issue") != "")  # Select rejected rows.
accepted_count = accepted_df.count()  # Count Silver candidates.
rejected_count = rejected_df.count()  # Count error candidates.
critical_count = checked_df.filter(F.col("critical_issue") != "").count()  # Count critical failures.
warning_count = accepted_df.filter(F.col("warning_issue") != "").count()  # Count accepted warnings.
if accepted_count + rejected_count != bronze_count:  # Account for each Bronze row.
    raise ValueError("Validation lost source rows")  # Stop if counts differ.
print("Accepted:", accepted_count, "Rejected:", rejected_count)  # Show the split.
print("Critical:", critical_count, "Warnings:", warning_count)  # Show severities.


Accepted: 6 Rejected: 0
Critical: 0 Warnings: 0


In [8]:
run_id = str(uuid4())  # Give this run a unique audit label.
if rejected_count > 0:  # Save evidence when invalid rows exist.
    error_path = project_root / "error" / "data" / table_name  # Choose the ignored error folder.
    error_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent exists.
    errors_df = (rejected_df.withColumn("run_id", F.lit(run_id))  # Identify this run.
                 .withColumn("rejected_at", F.current_timestamp()))  # Record processing time.
    errors_df.write.format("delta").mode("append").save(str(error_path))  # Keep earlier error runs.
    saved_errors_df = spark.read.format("delta").load(str(error_path))  # Read error evidence back.
    if saved_errors_df.filter(F.col("run_id") == run_id).count() != rejected_count:  # Verify this run.
        raise ValueError("Rejected rows were not saved correctly")  # Keep prior Silver intact.
if critical_count > 0:  # Apply workbook Critical / Fail pipeline rules.
    raise ValueError(f"{critical_count} critical failures; Silver was not overwritten")  # Stop safely.
silver_path = project_root / "lakehouse" / "silver" / "data" / table_name  # Choose the output path.
silver_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the parent exists.
output_columns = expected_columns + ["warning_issue"]  # Keep business fields and warning evidence.
silver_df = accepted_df.select(*output_columns)  # Omit raw JSON and helper fields.
silver_df.write.format("delta").mode("overwrite").save(str(silver_path))  # Save a fresh clean snapshot.
saved_df = spark.read.format("delta").load(str(silver_path))  # Read the committed Delta version.
if saved_df.count() != accepted_count or saved_df.columns != output_columns:  # Reconcile saved rows.
    raise ValueError("Silver save did not match the accepted dataset")  # Report a mismatch.
print("Bronze:", bronze_count, "Silver:", saved_df.count(), "Rejected:", rejected_count)  # Show counts.
print("Warnings in Silver:", saved_df.filter(F.col("warning_issue") != "").count())  # Show warnings.
print("Delta log exists:", (silver_path / "_delta_log").is_dir())  # Confirm Delta output.


Bronze: 6 Silver: 6 Rejected: 0
Warnings in Silver: 0
Delta log exists: True


In [9]:
saved_types = {f.name: f.dataType.simpleString() for f in saved_df.schema.fields}  # Inspect stored types.
for name, kind in fields:  # Check each typed business field.
    if kind not in ("text", "optional_text"):  # Text fields need no conversion check.
        expected_type = "int" if kind == "optional_int" else kind  # Normalize integer name.
        if saved_types[name] != expected_type:  # Compare actual and intended types.
            raise ValueError(f"{name}: expected {expected_type}, got {saved_types[name]}")  # Explain mismatch.
print("Saved types:", saved_types)  # Show the final Silver schema.


Saved types: {'store_id': 'string', 'store_name': 'string', 'city': 'string', 'region': 'string', 'latitude': 'decimal(9,6)', 'longitude': 'decimal(9,6)', 'opened_date': 'date', 'manager_name': 'string', 'warning_issue': 'string'}
